# Final Test — Independent Ground Truth Evaluation

Notebook ini menghitung evaluasi utama retrieval/ranking pada test set menggunakan
`ground_truth_relevance_test_v2.csv`, yaitu relevance set yang sudah ditentukan sebelum
hasil test final dilihat.

Konfigurasi sistem dibekukan:
- Query: `query_text_v2` / BASE_V2
- Semantic retrieval: Top-50
- Keyword retrieval tambahan: maksimal 20
- Semantic normalization: min-max per candidate pool
- WLC: `0.7 × semantic + 0.3 × harmonic efficiency`
- Test set: 90 query

Metrik utama:
- Candidate Hit Rate
- Hit@5
- Hit@10
- MRR

Notebook ini tidak melakukan tuning.


In [1]:
import pandas as pd
import numpy as np

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    TOP_K_SBERT,
    MAX_KEYWORD_EXTRA,
    W1_FINAL,
    W2_FINAL,
)

TEST_PATH = "ground_truth_relevance_test_v2.csv"
SUMMARY_OUT = "hasil_final_test_independent_groundtruth_ringkas.csv"
DETAIL_OUT = "hasil_final_test_independent_groundtruth_detail.csv"


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def reciprocal_rank_from_ids(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)
    for i, rid in enumerate(ranked_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / i
    return 0.0


# 1) Load test set
test = pd.read_csv(TEST_PATH)

required_cols = {"scraped_title", "query_text_v2", "relevant_ids"}
missing = required_cols - set(test.columns)
if missing:
    raise ValueError(f"Kolom test hilang: {sorted(missing)}")

dup_cases = test.duplicated(["query_text_v2", "relevant_ids"]).sum()
if dup_cases:
    raise ValueError(f"Test masih punya {dup_cases} duplicate query+relevance cases.")

print("VALIDASI TEST SET")
print(f"Jumlah query              : {len(test)}")
print(f"Duplicate query+relevance : {dup_cases}")

# 2) Init frozen production service
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

print("\nFROZEN CONFIG")
print(f"Semantic top-k            : {TOP_K_SBERT}")
print(f"Keyword extra max         : {MAX_KEYWORD_EXTRA}")
print(
    f"WLC                       : {W1_FINAL:.1f} semantic + {W2_FINAL:.1f} efficiency"
)
print("Semantic normalization    : min-max")
print("Efficiency                : harmonic mean")

# 3) Validate relevant IDs exist
all_relevant_ids = sorted(
    {rid for ids in test["relevant_ids"].apply(split_pipe) for rid in ids}
)

found = service.collection.get(ids=all_relevant_ids, include=[])
found_ids = set(found["ids"])
missing_ids = [rid for rid in all_relevant_ids if rid not in found_ids]

print("\nGROUND TRUTH VALIDATION")
print(f"Unique relevant IDs         : {len(all_relevant_ids)}")
print(f"Relevant IDs found in Chroma: {len(found_ids)}")

if missing_ids:
    raise ValueError(
        f"Ada {len(missing_ids)} relevant ID yang tidak tersedia di Chroma. "
        f"Contoh: {missing_ids[:10]}"
    )

# 4) Evaluate
rows = []
candidate_hit_count = 0
hit5_count = 0
hit10_count = 0
rr_values = []

print("\nMenjalankan frozen configuration pada test set...")

for _, row in test.iterrows():
    query_text = str(row["query_text_v2"]).strip()
    relevant_ids = set(split_pipe(row["relevant_ids"]))
    user_ingredients = parse_user_ingredients(query_text)

    candidates = service.retrieve_candidates(query_text, user_ingredients)
    candidate_ids = [str(c["id"]) for c in candidates]

    candidate_hit = int(any(rid in relevant_ids for rid in candidate_ids))
    candidate_hit_count += candidate_hit

    if candidates:
        semantic_scores = np.asarray(
            [float(c["sbert_score"]) for c in candidates],
            dtype=float,
        )
        s_min = float(semantic_scores.min())
        s_max = float(semantic_scores.max())
        semantic_norm = (semantic_scores - s_min) / (s_max - s_min + 1e-9)

        ranked = []
        for c, s_norm in zip(candidates, semantic_norm):
            stats = coverage_stats(user_ingredients, c["ingredient_names"])
            efficiency = harmonic_efficiency(
                stats["coverage_recipe"],
                stats["coverage_user"],
            )
            final_score = W1_FINAL * float(s_norm) + W2_FINAL * float(efficiency)
            ranked.append(
                {
                    **c,
                    "sbert_score_norm": float(s_norm),
                    "efficiency": float(efficiency),
                    "final_score": float(final_score),
                }
            )

        ranked.sort(
            key=lambda c: (
                -c["final_score"],
                -c["sbert_score_norm"],
                str(c["id"]),
            )
        )
    else:
        ranked = []

    ranked_ids = [str(c["id"]) for c in ranked]
    top5_ids = ranked_ids[:5]
    top10_ids = ranked_ids[:10]

    hit5 = int(any(rid in relevant_ids for rid in top5_ids))
    hit10 = int(any(rid in relevant_ids for rid in top10_ids))
    rr = reciprocal_rank_from_ids(ranked_ids, relevant_ids)

    hit5_count += hit5
    hit10_count += hit10
    rr_values.append(rr)

    best_rank = None
    best_title = None
    for rank, c in enumerate(ranked, start=1):
        if str(c["id"]) in relevant_ids:
            best_rank = rank
            best_title = c["title"]
            break

    rows.append(
        {
            "scraped_title": row["scraped_title"],
            "query_text": query_text,
            "relevant_ids": " | ".join(sorted(relevant_ids)),
            "candidate_pool_size": len(candidates),
            "candidate_hit": candidate_hit,
            "hit_at_5": hit5,
            "hit_at_10": hit10,
            "reciprocal_rank": rr,
            "best_relevant_rank": best_rank,
            "best_relevant_title": best_title,
            "top10_result_ids": " | ".join(top10_ids),
            "top10_result_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
        }
    )

detail_df = pd.DataFrame(rows)
n_query = len(test)

candidate_hit_rate = candidate_hit_count / n_query
hit_at_5 = hit5_count / n_query
hit_at_10 = hit10_count / n_query
mrr = float(np.mean(rr_values))

summary_df = pd.DataFrame(
    [
        {
            "n_query": n_query,
            "candidate_hit_rate": candidate_hit_rate,
            "candidate_hit_count": candidate_hit_count,
            "hit_at_5": hit_at_5,
            "hit_at_5_count": hit5_count,
            "hit_at_10": hit_at_10,
            "hit_at_10_count": hit10_count,
            "mrr": mrr,
            "semantic_top_k": TOP_K_SBERT,
            "keyword_extra_max": MAX_KEYWORD_EXTRA,
            "semantic_mode": "minmax",
            "w1": W1_FINAL,
            "w2": W2_FINAL,
        }
    ]
)

summary_df.to_csv(SUMMARY_OUT, index=False, encoding="utf-8-sig")
detail_df.to_csv(DETAIL_OUT, index=False, encoding="utf-8-sig")

print("\n" + "=" * 100)
print("HASIL FINAL TEST - INDEPENDENT GROUND TRUTH")
print("=" * 100)
print(
    f"Candidate Hit Rate : {candidate_hit_rate:.4f} ({candidate_hit_count}/{n_query})"
)
print(f"Hit@5              : {hit_at_5:.4f} ({hit5_count}/{n_query})")
print(f"Hit@10             : {hit_at_10:.4f} ({hit10_count}/{n_query})")
print(f"MRR                : {mrr:.4f}")

print("\nPENTING")
print(
    "- Ground truth berasal dari relevance set independen yang sudah dibuat sebelum final test dibuka."
)
print("- Tidak menggunakan manual qrels Top-10 untuk metrik utama ini.")
print("- Konfigurasi tidak boleh diubah lagi setelah hasil test ini dilihat.")

print("\nOUTPUT")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VALIDASI TEST SET
Jumlah query              : 90
Duplicate query+relevance : 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12259.18it/s]



FROZEN CONFIG
Semantic top-k            : 50
Keyword extra max         : 20
WLC                       : 0.7 semantic + 0.3 efficiency
Semantic normalization    : min-max
Efficiency                : harmonic mean

GROUND TRUTH VALIDATION
Unique relevant IDs         : 160
Relevant IDs found in Chroma: 160

Menjalankan frozen configuration pada test set...

HASIL FINAL TEST - INDEPENDENT GROUND TRUTH
Candidate Hit Rate : 0.6000 (54/90)
Hit@5              : 0.2111 (19/90)
Hit@10             : 0.3000 (27/90)
MRR                : 0.1485

PENTING
- Ground truth berasal dari relevance set independen yang sudah dibuat sebelum final test dibuka.
- Tidak menggunakan manual qrels Top-10 untuk metrik utama ini.
- Konfigurasi tidak boleh diubah lagi setelah hasil test ini dilihat.

OUTPUT
- hasil_final_test_independent_groundtruth_ringkas.csv
- hasil_final_test_independent_groundtruth_detail.csv
